# CSC2042S Assignment 1: Unsupervised Learning

**Student Number:** `FFLADA001` &nbsp;&nbsp; **Name:** `Adam Foflonker`

**High-level pipeline:**

1. Data Preprocessing
2. Missingness filtering, imputation and scaling
3. Exploratory visualization with t-SNE  
4. K-means Clustering implementation with **random** and **k-means++** initializations  
5. Convergence criteria tests
6. Determining the optimal **K**
7. Dimensionality reduction with **PCA**
8. Cluster Interpretation
---

## Import all required libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt 
%matplotlib inline

from pathlib import Path
from typing import Optional

## Define global constants, state and flags

In [ ]:
DATA_PATH: Path = Path("./data/WDICSV.csv")
RANDOM_STATE: int = 42

plt.rcParams.update({ "figure.figsize": (8, 5), "axes.grid": True })
np.random.seed(RANDOM_STATE)

# Feature name constants
FEATURE_BASE_COUNTRY_NAME: str = "Country Name"
FEATURE_BASE_COUNTRY_CODE: str = "Country Code"
FEATURE_BASE_INDICATOR_NAME: str = "Indicator Name"
FEATURE_BASE_INDICATOR_CODE: str = "Indicator Code"

FEATURE_BASE_COUNTRY_ALL: list[str] = [FEATURE_BASE_COUNTRY_NAME, FEATURE_BASE_COUNTRY_CODE]
FEATURE_BASE_INDICATOR_ALL: list[str] = [FEATURE_BASE_INDICATOR_NAME, FEATURE_BASE_INDICATOR_CODE]
FEATURE_BASE_ALL: list[str] = FEATURE_BASE_COUNTRY_ALL + FEATURE_BASE_INDICATOR_ALL

FEATURE_DERIVED_YEAR: str = "Year"

# Feature filtering thresholds
MIN_FEATURE_COVERAGE: int = 0.60  # Keep features with >= 60% (non-NaN)
MIN_SAMPLE_COVERAGE: int = 0.70   # Keep samples (country-year) with >= 70% (non-NaN)

# The best K-value to use for clustering as determined by the silhouette score in section 6
BEST_K: int = 4

## 1) Data Preprocessing

In [ ]:

def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Could not find \"{DATA_PATH}\"!")
    
    return pd.read_csv(path)

raw_df = load_dataset(DATA_PATH)
print(raw_df.shape)
raw_df.head(3)

### Reshape to country–year samples

Here we pivot such that each **row** is a **(Country, Year)** and each **column** is an **Indicator Code**.

In [ ]:

# Identify year columns (strings like '1960', ..., '2024')
year_cols = [c for c in raw_df.columns if c.isdigit()]

# Pivot to get a MultiIndex (Country, Year) by Indicator Code
# - Melt years long: columns -> rows (Year, Value)
# - Pivot with indicator code as columns
long_df = raw_df.melt(id_vars = FEATURE_BASE_ALL, value_vars = year_cols, 
                      var_name = FEATURE_DERIVED_YEAR, value_name = "Value")

long_df[FEATURE_DERIVED_YEAR] = long_df[FEATURE_DERIVED_YEAR].astype(int)

# Pivot: index = (Country Code, Country Name, Year), columns = Indicator Code
pivot_df = long_df.pivot_table(index = [FEATURE_BASE_COUNTRY_CODE, FEATURE_BASE_COUNTRY_NAME, FEATURE_DERIVED_YEAR], 
                            columns = FEATURE_BASE_INDICATOR_CODE, values = "Value", aggfunc = "mean").sort_index()

# Preserve indicator metadata for later interpretation
indicator_meta = raw_df[FEATURE_BASE_INDICATOR_ALL].drop_duplicates().set_index(FEATURE_BASE_INDICATOR_CODE)

print(pivot_df.shape)
pivot_df.head(3)

## 2) Missingness filtering, imputation and scaling

In [ ]:
def filter_by_missingness(pivot_df: pd.DataFrame, min_feat_cov: float, min_sample_cov: float) -> pd.DataFrame:
    # Feature coverage
    feat_non_na_frac = 1.0 - pivot_df.isna().mean(axis = 0)
    keep_feats = feat_non_na_frac[feat_non_na_frac >= min_feat_cov].index
    X = pivot_df[keep_feats].copy()

    # Sample coverage
    sample_non_na_frac = 1.0 - X.isna().mean(axis = 1)
    keep_samples = sample_non_na_frac[sample_non_na_frac >= min_sample_cov].index
    X = X.loc[keep_samples].copy()

    return X

X_filtered = filter_by_missingness(pivot_df, MIN_FEATURE_COVERAGE, MIN_SAMPLE_COVERAGE)
print("After filtering:", X_filtered.shape)

feat_coverage = 1.0 - X_filtered.isna().mean(axis = 0)
sample_coverage = 1.0 - X_filtered.isna().mean(axis = 1)

print("Median feature coverage:", round(float(feat_coverage.median()), 3))
print("Median sample coverage:", round(float(sample_coverage.median()), 3))

### Median imputation and scaling

In [ ]:
imputer = SimpleImputer(strategy = "median")  # Median imputation works better than 'mean' for outliers
scaler = StandardScaler()

X_imputed = pd.DataFrame(imputer.fit_transform(X_filtered), index = X_filtered.index, columns = X_filtered.columns)
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), index = X_imputed.index, columns = X_imputed.columns)

print("Shapes:")
print(" Filtered:", X_filtered.shape)
print(" Imputed:", X_imputed.shape)
print(" Scaled:", X_scaled.shape)
X_scaled.head(3)

## 3) Exploratory visualization with t-SNE 

In [ ]:
# Subsample for t-SNE since the data set is fairly large
N_TSNE: int = min(3000, len(X_scaled))
X_tsne_sample = X_scaled.sample(N_TSNE, random_state = RANDOM_STATE).values

tsne = TSNE(n_components = 2, random_state = RANDOM_STATE, init = "pca", perplexity = 50)
X_embedded = tsne.fit_transform(X_tsne_sample)

plt.figure()
plt.scatter(X_embedded[:,0], X_embedded[:, 1], s = 6, alpha = 0.7)
plt.title("t-SNE of WDI Dataset (subset)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

## 4) K-means Clustering implementation with **random** and **k-means++** initializations

In [ ]:
def kmeanspp_init(X: np.ndarray, k: int, rng: np.random.RandomState) -> np.ndarray:
    n, d = X.shape
    
    centroids = np.empty((k, d), dtype = X.dtype)
    idx = rng.randint(0, n)
    centroids[0] = X[idx]

    closest_sq = np.full(n, np.inf)
    for i in range(1, k):
        # Update closest squared distances
        dist_sq = ((X - centroids[i - 1]) ** 2).sum(axis = 1)
        closest_sq = np.minimum(closest_sq, dist_sq)
        probs = closest_sq / closest_sq.sum()
        
        cumulative = np.cumsum(probs)
        r = rng.rand()
        j = np.searchsorted(cumulative, r)
        centroids[i] = X[j]
        
    return centroids

def random_init(X: np.ndarray, k: int, rng: np.random.RandomState) -> np.ndarray:
    idx = rng.choice(X.shape[0], size = k, replace = False)
    return X[idx].copy()

def assign_labels(X: np.ndarray, C: np.ndarray):
    d2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis = 2)
    return d2.argmin(axis = 1), d2

def compute_centroids(X: np.ndarray, labels: np.ndarray, k: int) -> np.ndarray:
    d = X.shape[1]
    C = np.zeros((k, d), dtype = X.dtype)
    for j in range(k):
        boolMask = (labels == j)
        C[j] = X[boolMask].mean(axis = 0) if np.any(boolMask) else X[np.random.randint(0, len(X))]
            
    return C

def inertia_from_d2(labels: np.ndarray, d2: np.ndarray) -> float:
    rows = np.arange(len(labels))
    return float(d2[rows, labels].sum())

def kmeans(X: np.ndarray, k: int, init: str = "kmeans++", max_iters: int = 300, tol_centroid: float = 1e-4, 
           tol_inertia: float = 1e-6, early_stop_patience: Optional[int] = None, random_state: int = RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    
    if init == "kmeans++":
        C = kmeanspp_init(X, k, rng)
    elif init == "random":
        C = random_init(X, k, rng)
    else:
        raise ValueError("init must be 'kmeans++' or 'random'")

    inertia_history = []
    centroid_shift_history = []
    best = {"inertia": np.inf}
    no_improve_count = 0

    for it in range(1, max_iters + 1):
        labels, d2 = assign_labels(X, C)
        inertia = inertia_from_d2(labels, d2)
        inertia_history.append(inertia)

        C_new = compute_centroids(X, labels, k)
        centroid_shift = float(np.linalg.norm(C_new - C))
        centroid_shift_history.append(centroid_shift)

        if (inertia + 1e-12) < best.get("inertia", np.inf):
            best = {
                "centroids": C_new.copy(),
                "labels": labels.copy(),
                "inertia": float(inertia),
                "iters": it,
                "inertia_history": inertia_history.copy(),
                "centroid_shift_history": centroid_shift_history.copy()
            }
            
            no_improve_count = 0
        else:
            no_improve_count += 1

        # Convergence checks
        if centroid_shift < tol_centroid:
            break
        if (it > 1) and (abs(inertia_history[-2] - inertia_history[-1]) < tol_inertia):
            break
        if (early_stop_patience is not None) and (no_improve_count >= early_stop_patience):
            break

        C = C_new

    return best

## 5) Convergence criteria tests

In [ ]:
X_work = X_scaled.values

def run_convergence_suite(k = 6):
    results = {}

    # A) Max-iter test
    res_max = kmeans(X_work, k = k, init = "kmeans++", max_iters = 100, tol_centroid = 0.0, tol_inertia = 0.0)
    results["max_iters_only"] = res_max

    # B) Centroid shift tolerance test
    res_cent = kmeans(X_work, k = k, init = "kmeans++", max_iters = 300, tol_centroid = 1e-3, tol_inertia = 0.0)
    results["centroid_tol"] = res_cent

    # C) Inertia improvement tolerance test
    res_iner = kmeans(X_work, k = k, init = "kmeans++", max_iters = 300, tol_centroid = 0.0, tol_inertia = 1e-3)
    results["inertia_tol"] = res_iner

    # D) Early stopping patience test
    res_early = kmeans(X_work, k = k, init = "kmeans++", max_iters = 300, tol_centroid = 1e-6, tol_inertia = 1e-9,
                          early_stop_patience = 5)
    results["early_patience"] = res_early

    return results

conv_results = run_convergence_suite()
for name, res in conv_results.items():
    print(name, "-> iters:", res["iters"], ", inertia:", round(res["inertia"], 2))

# Plot objective trajectories
for name, res in conv_results.items():
    plt.figure()
    plt.plot(res["inertia_history"])
    plt.title(f"Inertia over iterations - {name}")
    plt.xlabel("Iteration")
    plt.ylabel("Inertia (WCSS)")
    plt.show()

# Plot centroid shifts
for name, res in conv_results.items():
    plt.figure()
    plt.plot(res["centroid_shift_history"])
    plt.title(f"Centroid shift over iterations - {name}")
    plt.xlabel("Iteration")
    plt.ylabel("L2 shift")
    plt.show()

## 6) Determining the optimal **K**

In [ ]:
def sweep_k(X: np.ndarray, k_min = 2, k_max = 15, init = "kmeans++") -> pd.DataFrame:
    rows = []
    for k in range(k_min, k_max + 1):
        res = kmeans(X, k = k, init = init, early_stop_patience = 10)
        labels = res["labels"]
        sil = np.nan
        
        if (len(np.unique(labels)) >= 2) and (len(np.unique(labels)) < len(labels)):
            try:
                sil = silhouette_score(X, labels)
            except Exception:
                sil = np.nan
                
        rows.append((k, res["inertia"], sil))
        
    return pd.DataFrame(rows, columns = ["k", "inertia", "silhouette"])

k_results = sweep_k(X_work, 2, 15, init = "kmeans++")
display(k_results)

# Elbow plot
plt.figure()
plt.plot(k_results["k"], k_results["inertia"], marker = "o")
plt.title("Elbow curve (WCSS vs K)")
plt.xlabel("K")
plt.ylabel("Inertia (WCSS)")
plt.show()

# Silhouette plot
plt.figure()
plt.plot(k_results["k"], k_results["silhouette"], marker = "o")
plt.title("Silhouette score vs K")
plt.xlabel("K")
plt.ylabel("Silhouette")
plt.show()

## 7) Dimensionality reduction with **PCA**

In [ ]:
pca = PCA(n_components = 14, random_state = RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled.values)

expl_var = pca.explained_variance_ratio_
print("Explained variance ratio (first 14):", np.round(expl_var, 4))

# Inspect top contributing original features for first 3 PCs
components = pca.components_
feature_names = X_scaled.columns.to_numpy()

def top_features_for_pc(pc_idx, topn = 8):
    weights = components[pc_idx]
    order = np.argsort(np.abs(weights))[-topn:][::-1]
    
    return list(zip(feature_names[order], np.round(weights[order], 4)))

for pc in range(3):
    print(f"Top features for PC{pc + 1}:")
    for name, w in top_features_for_pc(pc, topn = 14):
        readable = indicator_meta.loc[name][FEATURE_BASE_INDICATOR_NAME] if name in indicator_meta.index else name
        print("  ", name, " | ", readable, " | weight=", w)

res_orig = kmeans(X_scaled.values, k = BEST_K, init = "kmeans++", early_stop_patience = 10)
res_pca = kmeans(X_pca[:, :min(6, X_pca.shape[1])], k = BEST_K, init = "kmeans++", early_stop_patience = 10)

print("Inertia (original):", round(res_orig["inertia"], 2))
print("Inertia (PCA 6D):", round(res_pca["inertia"], 2))

# 2D PCA visualization
labels_pca = res_pca["labels"]
plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], s = 6, alpha = 0.7, c = labels_pca)
plt.title("Clusters in 2D PCA space (first 2 PCs)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# 3D PCA visualization
fig = plt.figure()
ax = fig.add_subplot(111, projection = '3d')
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], s = 6, alpha = 0.7, c = labels_pca)
ax.set_title("Clusters in 3D PCA space (PC1-PC3)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()

## 8) Cluster interpretation

In [ ]:
labels = labels_pca
profile = pd.DataFrame(X_scaled, index = X_scaled.index).groupby(labels).mean()
display(profile.head())

# Representative country-years per cluster (closest to centroids in PCA space)
def representative_samples(X, labels, centroids, topn = 7):
    reps = {}
    for j in np.unique(labels):
        idx = np.where(labels == j)[0]
        Xj = X[idx]
        d2 = ((Xj - centroids[j]) ** 2).sum(axis = 1)
        order = np.argsort(d2)[:topn]
        reps[j] = idx[order]
        
    return reps

reps = representative_samples(X_pca[:, :min(6, X_pca.shape[1])], labels_pca, res_pca["centroids"], topn = 7)
index_df = pd.DataFrame(list(X_scaled.index), columns = [FEATURE_BASE_COUNTRY_CODE, FEATURE_BASE_COUNTRY_NAME, FEATURE_DERIVED_YEAR])

for cl, idxs in reps.items():
    print(f"Cluster {cl} representatives:")
    for ridx in idxs:
        cc, cn, yr = index_df.iloc[ridx].tolist()
        print(f"  {cn} ({cc}) - {yr}")